In [ ]:
import sys
from pathlib import Path
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

project_root = Path.cwd()
while project_root.name != 'python' and project_root.parent != project_root:
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

## Downloading the data
#### Source: _Yahoo Finance_

In [ ]:
from data import download_tickers_history

# set the date range for the historic data
start_date = datetime(year=2020, month=1, day=1)
end_date = datetime(year=2025, month=12, day=31)
history = download_tickers_history(start_date, end_date, ['NVDA']);

nvda = history.NVDA;


## Autocorrelation and Market "Memory"
_Hypothesis_: Can tomorrow’s return be predicted by today’s return?

Will just use autocorrelation function with 1, 2, 3, 5, 10, 20 days shift

* Raw Log Returns ($r_t$): Measures directional predictability (found: $≈0$).
* Squared Log Returns ($r_t^2$): Measures variance memory (found: $+0.13$).
* Absolute Log Returns ($|r_t|$): Measures magnitude memory without outlier distortion


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf

prices = nvda['Close'].to_numpy()
log_returns = np.diff(np.log(prices))

clean_data = pd.DataFrame({
    'Log_Return': log_returns,
    'Squared_Return': log_returns ** 2,
    'Abs_Return': np.abs(log_returns)
}).dropna()

lags = [1, 2, 3, 5, 10, 20]
autocorr_results = []

for lag in lags:
    autocorr_results.append({
        'Lag': lag,
        'Raw Returns (Direction)': clean_data['Log_Return'].autocorr(lag),
        'Squared Returns (r^2)': clean_data['Squared_Return'].autocorr(lag),
        'Absolute Returns (|r|)': clean_data['Abs_Return'].autocorr(lag)
    })
results_df = pd.DataFrame(autocorr_results)
print(results_df)

print('--------------- \n')

# data visualization
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)

plot_acf(
    clean_data['Log_Return'],
    lags=30,
    ax=axes[0],
    title='Raw Returns r_t (No Memory)',
    color='r',
    alpha=0.05
)

plot_acf(
    clean_data['Squared_Return'],
    lags=30,
    ax=axes[1],
    title='Squared Returns r_t^2 (Volatility)',
    color='b',
    alpha=0.05
)

plot_acf(
    clean_data['Abs_Return'],
    lags=30,
    ax=axes[2],
    title='Absolute Returns |r_t| (Long Memory)',
    color='g',
    alpha=0.05
)

for ax in axes:
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.set_xlabel('Lag (Days)')

axes[0].set_ylabel('Autocorrelation Coefficient')
plt.tight_layout()

plt.show()


We can see larger numbers for squares and absolute returns. This demonstrates the ability of the market to have a "memory".
Despite the fact that the values differs not drastically, e.g. $|r_t - r_t^2| ≈ 0.04-0.07$ and $|r_t - |r_t|| ≈ 0.13-0.16$, this shows a statistically big difference.

 For $N ≈ 1,650$ trading days, the 95% noise boundary around zero is:
 $$±\frac{2}{\sqrt{N}} = ±\frac{2}{\sqrt{1650}} ≈ ±0.049$$
Thus, a value of $+0.16$ is more than $3.2$ standard deviations away from zero ($p < 0.001$).

 Why $|r_t|$ is higher and decays slower than r²ₜ (The Taylor Effect):
* Squaring returns ($r_t^2$) amplifies extreme outlier days (e.g., a $15%$ earnings gap squared is $0.0225$, which is $225 ×$ larger than a $1%$ day). These outliers distort Pearson correlation.
* Taking absolute returns (|r_t|) is robust to extreme spikes. empirical finance shows that $|r_t|$ consistently exhibits higher autocorrelation and longer memory than $r_t^2$ₜ.